In [11]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from course_notes import NOTES
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer


titles = [t for t , _ in NOTES]
texts = [f"{t}:{body}" for t , body in NOTES]
 
embedder = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=20 , random_state=0),
    Normalizer()
)
index = embedder.fit_transform(texts)


def retrive(questions , k = 3):
    q = embedder.transform([questions])[0]
    scores = index @ q
    return[(scores[i] , titles [i]) for i in np.argsort(-scores)[:k]]

retrive("What is one hot encoding")

[(np.float64(0.9834687518296914), 'One-hot encoding'),
 (np.float64(0.05194904825453153), 'Gini impurity'),
 (np.float64(0.04601339838891473), 'Bag of words')]

# PDF Using RAG


In [1]:
import re
import numpy as np
import pymupdf                      # was: import fitz  (same API, renamed)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

PDF = "ml_course_notes.pdf"

In [ ]:
doc = pymupdf.open(PDF)
pages = [(n, re.sub(r"[ \t]+", " ", page.get_text().strip()))
         for n, page in enumerate(doc, start=1)]
pages = [(n, t) for n, t in pages if t]        # drop pages with no text layer

print(f"{len(pages)} pages")

print(pages[0][1][:200])

3 pages
Machine Learning Course Notes
Train/test split
We always hold back part of the data as a test set. The model never sees it during training, so its
score on the test set is an honest estimate of how it


In [14]:
def chunk(pages, size=120, overlap=30):
    out = []
    for page_no, text in pages:
        sents = re.split(r"(?<=[.!?])\s+", text)
        cur, n = [], 0
        for s in sents:
            w = len(s.split())
            if n + w > size and cur:
                out.append((page_no, " ".join(cur)))
                tail, kept = [], 0                  # carry the tail into the next chunk
                for prev in reversed(cur):
                    if kept >= overlap: break
                    tail.insert(0, prev); kept += len(prev.split())
                cur, n = tail, kept
            cur.append(s); n += w
        if cur:
            out.append((page_no, " ".join(cur)))
    return out

chunks = chunk(pages)
print(f"{len(chunks)} chunks")
chunks

16 chunks


[(1,
  'Machine Learning Course Notes\nTrain/test split\nWe always hold back part of the data as a test set. The model never sees it during training, so its\nscore on the test set is an honest estimate of how it will do on new data. A typical split is 75 percent\nfor training and 25 percent for testing. Overfitting\nOverfitting is when a model memorises the training rows instead of learning the pattern. The sign is\na large gap between train accuracy and test accuracy, for example train 1.000 but test 0.771. A\nmodel that is perfect on training data but weak on new data has memorised, not learned.'),
 (1,
  'The sign is\na large gap between train accuracy and test accuracy, for example train 1.000 but test 0.771. A\nmodel that is perfect on training data but weak on new data has memorised, not learned. Cross-validation\nCross-validation splits the data into k folds, trains on k minus one of them and tests on the held-out\nfold, then rotates and averages. It gives a steadier, more hones

In [ ]:
pagenos = [p for p, _ in chunks]
texts   = [t for _, t in chunks]

embedder = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=min(20, len(texts) - 1), random_state=0),
    Normalizer()
)
index = embedder.fit_transform(texts)
print(index.shape)          # (16, 15)

(16, 15)


array([[ 4.50976019e-01, -1.86551027e-01,  5.07652613e-01,
         2.79040695e-01,  1.44650619e-01,  1.98798081e-01,
         2.94359673e-01,  8.30774562e-02,  1.75118260e-01,
         3.06850402e-01, -1.27475859e-01,  1.32710717e-01,
        -3.11014775e-01, -5.49193970e-03, -1.29631574e-01],
       [ 5.17944423e-01, -2.24001572e-01,  5.21653770e-01,
         2.69998073e-01,  1.69615234e-01,  3.05612098e-01,
         1.20391092e-01, -8.01760008e-02,  1.35754452e-01,
        -5.39076154e-02,  1.27974309e-01, -8.48909065e-02,
         2.78828650e-01,  1.59950736e-02,  2.65564523e-01],
       [ 6.01406703e-01, -3.47874087e-01,  2.86583535e-01,
        -7.94556662e-02, -9.11558192e-02,  2.05435530e-01,
        -3.85086962e-01, -1.32913864e-01, -1.49634253e-01,
        -3.24286816e-01,  4.67882397e-02, -8.06677848e-02,
         3.07463671e-02, -2.55944001e-03, -2.74186793e-01],
       [ 4.89976785e-01, -3.21162128e-01, -7.67765973e-02,
        -4.01244821e-01, -3.12188645e-01, -1.72774606

In [8]:
def retrieve(question, k=3, min_score=0.3):
    q = embedder.transform([question])[0]
    scores = index @ q
    return [(scores[i], pagenos[i], texts[i])
            for i in np.argsort(-scores)[:k] if scores[i] >= min_score]

In [25]:
retrieve("Semantic")

[(np.float64(0.9952359293447414),
  3,
  'We use direction rather\nthan distance so document length does not matter. Semantic search\nSemantic search embeds every document once, embeds the incoming query the same way, scores\neach document by cosine similarity, and returns the highest scoring ones. It finds matches by\nmeaning rather than by exact keyword.'),
 (np.float64(0.7405544113453364),
  3,
  'Embeddings\nAn embedding represents a word or document as a short list of numbers, positioned so that similar\nmeanings sit close together. Unlike bag of words, it captures that good and great are related. Relationships become arithmetic, for example king minus man plus woman equals queen. Cosine similarity\nCosine similarity compares the direction of two vectors, computed as their dot product divided by\nboth their lengths. One means the same meaning, zero means unrelated. We use direction rather\nthan distance so document length does not matter. Semantic search\nSemantic search embeds ev